In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


data = [
    (1, "2025-01-01", "P1", ["milk", "bread"], {"store": "S1", "city": "Hyderabad"}, 500),
    (2, "2025-01-01", "P2", ["bread"], {"store": "S2", "city": "Delhi"}, 300),
    (3, "2025-01-02", "P1", ["milk", "butter"], {"store": "S1", "city": "Hyderabad"}, 700),
    (4, "2025-01-02", "P3", ["cheese", "milk"], {"store": "S3", "city": "Mumbai"}, 450),
    (5, "2025-01-03", "P2", ["bread", "butter"], {"store": "S2", "city": "Delhi"}, 600),
]

schema = StructType([
    StructField("txn_id", IntegerType()),
    StructField("date", StringType()),
    StructField("product_id", StringType()),
    StructField("basket_items", ArrayType(StringType())),
    StructField("store_info", MapType(StringType(), StringType())),
    StructField("revenue", IntegerType())
])

df = spark.createDataFrame(data, schema)
df.display()

txn_id,date,product_id,basket_items,store_info,revenue
1,2025-01-01,P1,"List(milk, bread)","Map(store -> S1, city -> Hyderabad)",500
2,2025-01-01,P2,List(bread),"Map(store -> S2, city -> Delhi)",300
3,2025-01-02,P1,"List(milk, butter)","Map(store -> S1, city -> Hyderabad)",700
4,2025-01-02,P3,"List(cheese, milk)","Map(store -> S3, city -> Mumbai)",450
5,2025-01-03,P2,"List(bread, butter)","Map(store -> S2, city -> Delhi)",600


## Explode Array (Market Basket Analysis Base)

In [0]:
#Goal is one row per item
df_exploded = df.withColumn("item",F.explode(F.col('basket_items')))
df_exploded.display()

txn_id,date,product_id,basket_items,store_info,revenue,item
1,2025-01-01,P1,"List(milk, bread)","Map(store -> S1, city -> Hyderabad)",500,milk
1,2025-01-01,P1,"List(milk, bread)","Map(store -> S1, city -> Hyderabad)",500,bread
2,2025-01-01,P2,List(bread),"Map(store -> S2, city -> Delhi)",300,bread
3,2025-01-02,P1,"List(milk, butter)","Map(store -> S1, city -> Hyderabad)",700,milk
3,2025-01-02,P1,"List(milk, butter)","Map(store -> S1, city -> Hyderabad)",700,butter
4,2025-01-02,P3,"List(cheese, milk)","Map(store -> S3, city -> Mumbai)",450,cheese
4,2025-01-02,P3,"List(cheese, milk)","Map(store -> S3, city -> Mumbai)",450,milk
5,2025-01-03,P2,"List(bread, butter)","Map(store -> S2, city -> Delhi)",600,bread
5,2025-01-03,P2,"List(bread, butter)","Map(store -> S2, city -> Delhi)",600,butter


## Collect List (Reverse Explode)

In [0]:
df_exploded.groupBy('product_id').agg(F.collect_list('item').alias('items')).display()

product_id,items
P1,"List(milk, bread, milk, butter)"
P2,"List(bread, bread, butter)"
P3,"List(cheese, milk)"


In [0]:
df_exploded.groupBy('product_id').agg(F.collect_set('item').alias('items')).display()

product_id,items
P3,"List(cheese, milk)"
P2,"List(bread, butter)"
P1,"List(bread, milk, butter)"


⚠ collect_list preserves duplicates
⚠ collect_set removes duplicates

In [0]:
df.display()

txn_id,date,product_id,basket_items,store_info,revenue
1,2025-01-01,P1,"List(milk, bread)","Map(store -> S1, city -> Hyderabad)",500
2,2025-01-01,P2,List(bread),"Map(store -> S2, city -> Delhi)",300
3,2025-01-02,P1,"List(milk, butter)","Map(store -> S1, city -> Hyderabad)",700
4,2025-01-02,P3,"List(cheese, milk)","Map(store -> S3, city -> Mumbai)",450
5,2025-01-03,P2,"List(bread, butter)","Map(store -> S2, city -> Delhi)",600


## Extract from MapType

In [0]:
df.select('txn_id',
          F.col('store_info')["store"].alias("store"),
          F.col('store_info')["city"].alias("city")
          ).display()

txn_id,store,city
1,S1,Hyderabad
2,S2,Delhi
3,S1,Hyderabad
4,S3,Mumbai
5,S2,Delhi


## Create Struct Column

In [0]:
df = df.withColumn(
    "store_struct",
    F.struct(
        F.col("store_info")["store"].alias("store"),
        F.col("store_info")["city"].alias("city")
    )
)

df.select("txn_id", "store_struct").display()

txn_id,store_struct
1,"List(S1, Hyderabad)"
2,"List(S2, Delhi)"
3,"List(S1, Hyderabad)"
4,"List(S3, Mumbai)"
5,"List(S2, Delhi)"


## Revenue per product per date

In [0]:
df.groupBy('date').pivot('product_id').agg(F.sum("revenue")).display()

date,P1,P2,P3
2025-01-01,500,300,null
2025-01-02,700,null,450
2025-01-03,null,600,null


## Advanced Feature Engineering with Arrays

In [0]:
df.withColumn("has_milk", F.array_contains("basket_items", "milk")).display()

txn_id,date,product_id,basket_items,store_info,revenue,store_struct,has_milk
1,2025-01-01,P1,"List(milk, bread)","Map(store -> S1, city -> Hyderabad)",500,"List(S1, Hyderabad)",true
2,2025-01-01,P2,List(bread),"Map(store -> S2, city -> Delhi)",300,"List(S2, Delhi)",false
3,2025-01-02,P1,"List(milk, butter)","Map(store -> S1, city -> Hyderabad)",700,"List(S1, Hyderabad)",true
4,2025-01-02,P3,"List(cheese, milk)","Map(store -> S3, city -> Mumbai)",450,"List(S3, Mumbai)",true
5,2025-01-03,P2,"List(bread, butter)","Map(store -> S2, city -> Delhi)",600,"List(S2, Delhi)",false


## Higher Order Functions (Advanced) - Filter items inside array

In [0]:
df.withColumn(
  "Milk_related",
  F.expr("filter(basket_items, x -> x = 'milk')")
).display()

txn_id,date,product_id,basket_items,store_info,revenue,store_struct,Milk_related
1,2025-01-01,P1,"List(milk, bread)","Map(store -> S1, city -> Hyderabad)",500,"List(S1, Hyderabad)",List(milk)
2,2025-01-01,P2,List(bread),"Map(store -> S2, city -> Delhi)",300,"List(S2, Delhi)",List()
3,2025-01-02,P1,"List(milk, butter)","Map(store -> S1, city -> Hyderabad)",700,"List(S1, Hyderabad)",List(milk)
4,2025-01-02,P3,"List(cheese, milk)","Map(store -> S3, city -> Mumbai)",450,"List(S3, Mumbai)",List(milk)
5,2025-01-03,P2,"List(bread, butter)","Map(store -> S2, city -> Delhi)",600,"List(S2, Delhi)",List()


In [0]:
df.withColumn(
    "upper_items",
   F.expr("transform(basket_items, x -> upper(x))")
).display()

txn_id,date,product_id,basket_items,store_info,revenue,store_struct,upper_items
1,2025-01-01,P1,"List(milk, bread)","Map(store -> S1, city -> Hyderabad)",500,"List(S1, Hyderabad)","List(MILK, BREAD)"
2,2025-01-01,P2,List(bread),"Map(store -> S2, city -> Delhi)",300,"List(S2, Delhi)",List(BREAD)
3,2025-01-02,P1,"List(milk, butter)","Map(store -> S1, city -> Hyderabad)",700,"List(S1, Hyderabad)","List(MILK, BUTTER)"
4,2025-01-02,P3,"List(cheese, milk)","Map(store -> S3, city -> Mumbai)",450,"List(S3, Mumbai)","List(CHEESE, MILK)"
5,2025-01-03,P2,"List(bread, butter)","Map(store -> S2, city -> Delhi)",600,"List(S2, Delhi)","List(BREAD, BUTTER)"


## Total revenue per store from MapType

In [0]:
df.groupBy(F.col('store_info')['store'].alias('store_info')).agg(F.sum('revenue')).display()

store_info,sum(revenue)
S1,1200
S2,900
S3,450


## Create revenue per city per product pivot table

In [0]:
df.groupBy(F.col('store_info')['city'].alias('City')).pivot('product_id').agg(F.sum('revenue')).display()

City,P1,P2,P3
Hyderabad,1200,null,null
Delhi,null,900,null
Mumbai,null,null,450
